# Gold Layer: UC Metric Views (Parker Aerospace)

Standardized KPIs for the Parker installed-base platform, in `genie_zeroops_mfg_catalog.default`:
- **component_health_metrics** (`gold_component_health`): health index, failure risk tier, warranty exposure, compliance
- **part_failure_metrics** (`ml_component_failure_forecast`): next-quarter forecast KPIs from the leakage-free point-in-time model (MLflow run `6b20bf8b32c84774807ed160317fa799`). This view previously read `ml_part_failure_predictions`, which was retired for target leakage (0.999 AUC).
- **maintenance_anomaly_metrics** (`gold_maintenance_anomalies`): anomaly frequency, severity, detection latency, warranty impact, MRO quality

The definitions below have the same sources, dimensions and measures as the live views on 2026-11-02 (some comments and display names are shortened). `part_failure_metrics` uses `ALTER VIEW` so its grants are preserved. Re-running a `CREATE OR REPLACE` cell recreates that view and resets its grants. The last cell validates all three views.

In [0]:
CREATE OR REPLACE VIEW genie_zeroops_mfg_catalog.default.component_health_metrics
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
source: genie_zeroops_mfg_catalog.default.gold_component_health
comment: "Parker component health KPIs: fleet-wide health index, failure risk distribution, warranty exposure, and maintenance compliance by airline and component type."
dimensions:
  - name: airline
    expr: airline
    comment: Airline customer operating the component
    synonyms: [carrier, customer]
  - name: component_type
    expr: component_type
    comment: Parker component category
    synonyms: [part type, component category]
  - name: criticality
    expr: criticality
    comment: FLIGHT_CRITICAL or ESSENTIAL
  - name: failure_risk_category
    expr: failure_risk_category
    comment: Current health-based risk tier (LOW/MEDIUM/HIGH/CRITICAL)
    synonyms: [risk level, risk tier]
  - name: parker_division
    expr: parker_division
    comment: Parker Hannifin business division
  - name: aircraft_type
    expr: aircraft_type
    comment: Aircraft model designation
measures:
  - name: Total Components
    expr: COUNT(1)
    comment: Number of installed Parker components
  - name: Avg Health Index
    expr: AVG(health_index)
    comment: Average component health score
    format: {type: number, decimal_places: {type: exact, places: 3}}
  - name: Critical Risk Components
    expr: COUNT(1) FILTER (WHERE failure_risk_category = 'CRITICAL')
  - name: High Risk Components
    expr: "COUNT(1) FILTER (WHERE failure_risk_category IN ('CRITICAL', 'HIGH'))"
    display_name: High + Critical Components
  - name: Active Anomalies
    expr: COUNT(1) FILTER (WHERE has_active_anomaly = true)
  - name: Total Warranty Exposure
    expr: SUM(estimated_warranty_exposure_usd)
    display_name: Total Warranty Exposure ($)
    format: {type: number, decimal_places: {type: exact, places: 0}, abbreviation: compact}
  - name: Avg Days to Failure
    expr: AVG(days_to_predicted_failure)
    display_name: Avg Days to Predicted Failure
    format: {type: number, decimal_places: {type: exact, places: 0}}
  - name: Avg Compliance Rate
    expr: AVG(maintenance_compliance_rate)
    format: {type: percentage, decimal_places: {type: exact, places: 1}}
  - name: Total Anomaly Events
    expr: SUM(anomaly_event_count)
$$

In [0]:
-- Repointed from the retired ml_part_failure_predictions (target leakage) to the leakage-free forecast.
-- Use ALTER VIEW so existing grants are kept; on a fresh workspace use CREATE OR REPLACE VIEW ... WITH METRICS LANGUAGE YAML.
ALTER VIEW genie_zeroops_mfg_catalog.default.part_failure_metrics AS $$
version: 1.1
source: genie_zeroops_mfg_catalog.default.ml_component_failure_forecast
comment: "Next-quarter component failure forecast KPIs from the leakage-free point-in-time model (MLflow run 6b20bf8b32c84774807ed160317fa799). Replaces the retired ml_part_failure_predictions model."
dimensions:
  - expr: airline
  - expr: component_type
  - expr: criticality
  - expr: recommended_action
  - expr: risk_decile
    comment: Forecast risk decile, 10 = highest-risk 10% of components
measures:
  - name: Total Components
    expr: COUNT(1)
  - name: Avg Forecast Probability
    expr: AVG(forecast_probability)
    format: {type: percentage, decimal_places: {type: exact, places: 1}}
  - name: Top-Decile Components
    expr: COUNT(1) FILTER (WHERE risk_decile = 10)
  - name: Proactive-Visit Components
    expr: COUNT(1) FILTER (WHERE recommended_action = 'PROACTIVE_FIELD_SERVICE_VISIT')
  - name: Expected Warranty Exposure
    expr: SUM(expected_warranty_exposure_usd)
    format: {type: currency, currency_code: USD, abbreviation: compact}
  - name: Next-Quarter Events (back-test)
    expr: SUM(actual_next_quarter_event)
    comment: Components that actually had a HIGH/CRITICAL or warranty-risk event in the forecast quarter
$$

In [0]:
CREATE OR REPLACE VIEW genie_zeroops_mfg_catalog.default.maintenance_anomaly_metrics
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
source: genie_zeroops_mfg_catalog.default.gold_maintenance_anomalies
comment: "Maintenance anomaly analysis KPIs: anomaly frequency by type, severity distribution, detection latency, warranty impact, and MRO quality tracking."
dimensions:
  - expr: airline
  - expr: component_type
  - name: anomaly_type
    expr: anomaly_type
    synonyms: [anomaly category, defect type]
  - name: anomaly_severity
    expr: anomaly_severity
    synonyms: [severity, severity level]
  - expr: flight_delay_risk
  - expr: mro_shop
  - expr: criticality
  - expr: parker_division
measures:
  - name: Total Anomalies
    expr: COUNT(1)
  - name: Critical Anomalies
    expr: COUNT(1) FILTER (WHERE anomaly_severity = 'CRITICAL')
  - name: High Delay Risk
    expr: "COUNT(1) FILTER (WHERE flight_delay_risk IN ('CRITICAL', 'HIGH'))"
  - name: Avg Days Undetected
    expr: AVG(days_undetected)
    format: {type: number, decimal_places: {type: exact, places: 1}}
  - name: Warranty Risk Events
    expr: COUNT(1) FILTER (WHERE warranty_claim_risk = true)
  - name: Total Warranty Exposure
    expr: SUM(record_warranty_exposure_usd)
    format: {type: number, decimal_places: {type: exact, places: 0}, abbreviation: compact}
  - name: Overdue Events
    expr: COUNT(1) FILTER (WHERE is_overdue = true)
  - name: Avg Overdue Days
    expr: AVG(overdue_days) FILTER (WHERE overdue_days > 0)
    format: {type: number, decimal_places: {type: exact, places: 0}}
$$

In [0]:
SELECT 'component_health_metrics' AS metric_view, MEASURE(`Total Components`) AS value, 'components' AS unit FROM genie_zeroops_mfg_catalog.default.component_health_metrics
UNION ALL
SELECT 'part_failure_metrics', MEASURE(`Proactive-Visit Components`), 'proactive-visit components' FROM genie_zeroops_mfg_catalog.default.part_failure_metrics
UNION ALL
SELECT 'part_failure_metrics', MEASURE(`Top-Decile Components`), 'top-decile components' FROM genie_zeroops_mfg_catalog.default.part_failure_metrics
UNION ALL
SELECT 'maintenance_anomaly_metrics', MEASURE(`Total Anomalies`), 'anomalies' FROM genie_zeroops_mfg_catalog.default.maintenance_anomaly_metrics